# Task 2 — Multiple Matrix Operations using OpenMP
**6CS005 High Performance Computing**

Covers: addition, subtraction, element-wise ×/÷, transpose, matrix multiplication — all parallelised with OpenMP.

## Step 1 — Check OpenMP availability

In [ ]:
!gcc --version
!echo '#include <omp.h>\nint main(){return 0;}' > /tmp/omp_test.c && gcc -fopenmp /tmp/omp_test.c -o /tmp/omp_test && echo '✅ OpenMP available'

## Step 2 — Write the C source file

In [ ]:
%%writefile matrix_ops.c
/*
 * Task 2: Multiple Matrix Operations with OpenMP
 * 6CS005 High Performance Computing
 * Compile: gcc -O2 -fopenmp -o matrix_ops matrix_ops.c -lm
 * Usage:   ./matrix_ops <input_file> <num_threads>
 */
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <math.h>
#include <omp.h>

typedef struct { int rows,cols; double **data; } Matrix;

static Matrix *alloc_matrix(int r,int c){
    Matrix*m=(Matrix*)malloc(sizeof(Matrix)); m->rows=r; m->cols=c;
    m->data=(double**)malloc(r*sizeof(double*));
    for(int i=0;i<r;i++) m->data[i]=(double*)calloc(c,sizeof(double));
    return m;
}

static void free_matrix(Matrix*m){
    if(!m)return;
    for(int i=0;i<m->rows;i++) free(m->data[i]);
    free(m->data); free(m);
}

static void write_matrix(FILE*f,const Matrix*m){
    for(int i=0;i<m->rows;i++){
        for(int j=0;j<m->cols;j++){if(j)fprintf(f,",");if(isnan(m->data[i][j]))fprintf(f,"NaN");else fprintf(f,"%.6g",m->data[i][j]);}
        fprintf(f,"\n");
    }
}

static void whdr(FILE*f,const char*op,int r,int c){fprintf(f,"\n--- %s (%d,%d) ---\n",op,r,c);}

static Matrix *read_matrix(FILE*fp){
    int r,c; if(fscanf(fp," %d %d",&r,&c)!=2)return NULL;
    if(r<=0||c<=0){fprintf(stderr,"[ERROR] Bad dims %dx%d\n",r,c);return NULL;}
    Matrix*m=alloc_matrix(r,c);
    for(int i=0;i<r;i++) for(int j=0;j<c;j++) if(fscanf(fp," %lf",&m->data[i][j])!=1){fprintf(stderr,"[ERROR] Missing data\n");free_matrix(m);return NULL;}
    return m;
}

static int cap(int req,int mx){return(req>mx)?mx:req;}

static void process_pair(FILE*out,int pi,Matrix*A,Matrix*B,int nt){
    int same=(A->rows==B->rows&&A->cols==B->cols);
    int mult=(A->cols==B->rows);
    fprintf(out,"\n========================================\nPAIR %d  A:%dx%d  B:%dx%d\n========================================\n",
            pi,A->rows,A->cols,B->rows,B->cols);
    fprintf(out,"\nMatrix A (%d,%d):\n",A->rows,A->cols); write_matrix(out,A);
    fprintf(out,"\nMatrix B (%d,%d):\n",B->rows,B->cols); write_matrix(out,B);

    /* Addition */
    if(same){Matrix*C=alloc_matrix(A->rows,A->cols);int t=cap(nt,A->rows);
        #pragma omp parallel for num_threads(t) schedule(static)
        for(int i=0;i<A->rows;i++) for(int j=0;j<A->cols;j++) C->data[i][j]=A->data[i][j]+B->data[i][j];
        whdr(out,"Addition",C->rows,C->cols);write_matrix(out,C);free_matrix(C);
    }else fprintf(out,"\n--- Addition ---\nAddition cannot be done (shapes differ).\n");

    /* Subtraction */
    if(same){Matrix*C=alloc_matrix(A->rows,A->cols);int t=cap(nt,A->rows);
        #pragma omp parallel for num_threads(t) schedule(static)
        for(int i=0;i<A->rows;i++) for(int j=0;j<A->cols;j++) C->data[i][j]=A->data[i][j]-B->data[i][j];
        whdr(out,"Subtraction",C->rows,C->cols);write_matrix(out,C);free_matrix(C);
    }else fprintf(out,"\n--- Subtraction ---\nSubtraction cannot be done (shapes differ).\n");

    /* Element-wise multiply */
    if(same){Matrix*C=alloc_matrix(A->rows,A->cols);int t=cap(nt,A->rows);
        #pragma omp parallel for num_threads(t) schedule(static)
        for(int i=0;i<A->rows;i++) for(int j=0;j<A->cols;j++) C->data[i][j]=A->data[i][j]*B->data[i][j];
        whdr(out,"Element-wise Multiplication (A.*B)",C->rows,C->cols);write_matrix(out,C);free_matrix(C);
    }else fprintf(out,"\n--- Element-wise Multiplication (A.*B) ---\nElement-wise multiplication cannot be done (shapes differ).\n");

    /* Element-wise divide */
    if(same){Matrix*C=alloc_matrix(A->rows,A->cols);int t=cap(nt,A->rows);
        #pragma omp parallel for num_threads(t) schedule(static)
        for(int i=0;i<A->rows;i++) for(int j=0;j<A->cols;j++) C->data[i][j]=(B->data[i][j]==0.0)?NAN:(A->data[i][j]/B->data[i][j]);
        whdr(out,"Element-wise Division (A./B)",C->rows,C->cols);write_matrix(out,C);free_matrix(C);
    }else fprintf(out,"\n--- Element-wise Division (A./B) ---\nElement-wise division cannot be done (shapes differ).\n");

    /* Transpose A */
    {Matrix*C=alloc_matrix(A->cols,A->rows);int t=cap(nt,A->rows);
        #pragma omp parallel for num_threads(t) schedule(static)
        for(int i=0;i<A->rows;i++) for(int j=0;j<A->cols;j++) C->data[j][i]=A->data[i][j];
        whdr(out,"Transpose of A (A^T)",C->rows,C->cols);write_matrix(out,C);free_matrix(C);}

    /* Transpose B */
    {Matrix*C=alloc_matrix(B->cols,B->rows);int t=cap(nt,B->rows);
        #pragma omp parallel for num_threads(t) schedule(static)
        for(int i=0;i<B->rows;i++) for(int j=0;j<B->cols;j++) C->data[j][i]=B->data[i][j];
        whdr(out,"Transpose of B (B^T)",C->rows,C->cols);write_matrix(out,C);free_matrix(C);}

    /* Matrix multiply A×B */
    if(mult){Matrix*C=alloc_matrix(A->rows,B->cols);int t=cap(nt,A->rows);
        #pragma omp parallel for num_threads(t) schedule(static)
        for(int i=0;i<A->rows;i++) for(int k=0;k<A->cols;k++) for(int j=0;j<B->cols;j++) C->data[i][j]+=A->data[i][k]*B->data[k][j];
        whdr(out,"Matrix Multiplication (A x B)",C->rows,C->cols);write_matrix(out,C);free_matrix(C);
    }else fprintf(out,"\n--- Matrix Multiplication (A x B) ---\nMatrix multiplication cannot be done (A.cols != B.rows).\n");
    fprintf(out,"\n");
}

int main(int argc,char*argv[]){
    if(argc<3){fprintf(stderr,"Usage: %s <input_file> <num_threads>\n",argv[0]);return 1;}
    const char*infile=argv[1]; int nt=atoi(argv[2]);
    if(nt<1){fprintf(stderr,"[ERROR] threads>=1\n");return 1;}
    FILE*fp=fopen(infile,"r"); if(!fp){perror("fopen input");return 1;}
    FILE*out=fopen("results.txt","w"); if(!out){perror("fopen results.txt");fclose(fp);return 1;}
    fprintf(out,"=== Matrix Operations Results ===\nInput: %s  Threads: %d\n",infile,nt);
    double t0=omp_get_wtime();
    int pair=1;
    while(1){Matrix*A=read_matrix(fp);if(!A)break;Matrix*B=read_matrix(fp);if(!B){fprintf(stderr,"[WARN] Odd matrix count\n");free_matrix(A);break;}
        process_pair(out,pair,A,B,nt);free_matrix(A);free_matrix(B);pair++;}
    fprintf(out,"\n=== Done: %d pair(s) in %.4fs ===\n",pair-1,omp_get_wtime()-t0);
    fclose(fp);fclose(out);
    printf("[INFO] %d pair(s) -> results.txt  (%.4fs)\n",pair-1,omp_get_wtime()-t0);
    return 0;
}

## Step 3 — Compile with OpenMP

In [ ]:
!gcc -O2 -fopenmp -o matrix_ops matrix_ops.c -lm && echo '✅ Compilation successful'

## Step 4 — Create sample input file

In [ ]:
matrices_input = """3 3
1 2 3
4 5 6
7 8 9
3 3
9 8 7
6 5 4
3 2 1
2 3
1 2 3
4 5 6
3 2
7 8
9 10
11 12
3 3
1 0 2
3 4 0
0 5 6
3 3
1 2 0
4 0 3
0 6 7
"""

with open('matrices.txt', 'w') as f:
    f.write(matrices_input)
print('matrices.txt created')

## Step 5 — Run the programme

In [ ]:
import subprocess
for nt in [1, 4]:
    r = subprocess.run(['./matrix_ops', 'matrices.txt', str(nt)], capture_output=True, text=True)
    print(f'--- {nt} thread(s) ---')
    print(r.stdout)
    if r.stderr: print('STDERR:', r.stderr)

## Step 6 — Display results.txt

In [ ]:
with open('results.txt') as f:
    print(f.read())